# Treinamento com interface de alto nível

## Importação das bibliotecas

In [1]:
# http://pytorch.org/
from os.path import exists

import torch

In [2]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Criação da rede

In [3]:
net_input = 28*28 #784
net_output = 10 # 10 dígitos

In [4]:
class Net(nn.Module):
    def __init__(self, net_input, net_output):
        super(Net, self).__init__()
        self.layer1 = nn.Linear(net_input, 1024)
        self.layer2 = nn.Linear(1024, 2048)
        self.layer3 = nn.Linear(2048, 4096)
        self.layer4 = nn.Linear(4096, 8192)
        self.layer5 = nn.Linear(8192, 4096)
        self.layer6 = nn.Linear(4096, 2048)
        self.layer7 = nn.Linear(2048, 1024)
        self.layer8 = nn.Linear(1024, net_output)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.layer1(x)
        x = F.relu(x)
        x = self.layer2(x)
        x = F.relu(x)
        x = self.layer3(x)
        x = F.relu(x)
        x = self.layer4(x)
        x = F.relu(x)
        x = self.layer5(x)
        x = F.relu(x)
        x = self.layer6(x)
        x = F.relu(x)
        x = self.layer7(x)
        x = F.relu(x)
        x = self.layer8(x)
        output = F.log_softmax(x, dim=1)
        return output

model = Net(net_input, net_output)

In [5]:
print(model)

Net(
  (layer1): Linear(in_features=784, out_features=1024, bias=True)
  (layer2): Linear(in_features=1024, out_features=2048, bias=True)
  (layer3): Linear(in_features=2048, out_features=4096, bias=True)
  (layer4): Linear(in_features=4096, out_features=8192, bias=True)
  (layer5): Linear(in_features=8192, out_features=4096, bias=True)
  (layer6): Linear(in_features=4096, out_features=2048, bias=True)
  (layer7): Linear(in_features=2048, out_features=1024, bias=True)
  (layer8): Linear(in_features=1024, out_features=10, bias=True)
)


Layer 1 ($784 \to 1024$):$(784 \times 1024) + 1024 = \mathbf{803.840}$

Layer 2 ($1024 \to 2048$):$(1024 \times 2048) + 2048 = \mathbf{2.099.200}$

Layer 3 ($2048 \to 4096$):$(2048 \times 4096) + 4096 = \mathbf{8.392.704}$

Layer 4 ($4096 \to 8192$):$(4096 \times 8192) + 8192 = \mathbf{33.562.624}$

Layer 5 ($8192 \to 4096$):$(8192 \times 4096) + 4096 = \mathbf{33.558.528}$

Layer 6 ($4096 \to 2048$):$(4096 \times 2048) + 2048 = \mathbf{8.390.656}$

Layer 7 ($2048 \to 1024$):$(2048 \times 1024) + 1024 = \mathbf{2.098.176}$

Layer 8 ($1024 \to 10$):$(1024 \times 10) + 10 = \mathbf{10.250}$

Total: 88.915.978 parâmetros (Quase 89 milhões).

**Rede anterior convolucional**
$$320 + 18.496 + 1.179.776 + 1.290 = \mathbf{1.199.882}$$Aproximadamente 1,2 milhão de parâmetros.


## Treinamento

### Criando o objeto de treinamento

In [6]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [7]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    acc = 100. * correct / len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        acc))
    return acc

## Avaliação

In [8]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 64}
test_kwargs = {'batch_size': 1000}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset1 = datasets.KMNIST('../data', train=True, download=True,
                    transform=transform)
dataset2 = datasets.KMNIST('../data', train=False,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net(net_input, net_output).to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1)

epochs = 5
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
best_acc = test(model, device, test_loader)

for epoch in range(1, epochs + 1):
    train(10, False, model, device, train_loader, optimizer, epoch)
    acc = test(model, device, test_loader)
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "kmnist_cnn.pt")
    scheduler.step()

100%|██████████| 18.2M/18.2M [00:09<00:00, 1.83MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 339kB/s]
100%|██████████| 3.04M/3.04M [00:01<00:00, 1.87MB/s]
100%|██████████| 5.12k/5.12k [00:00<00:00, 17.8MB/s]



Test set: Average loss: 2.3026, Accuracy: 1000/10000 (10%)

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.302608
Train Epoch: 1 [640/60000 (1%)]	Loss: 2.302780
Train Epoch: 1 [1280/60000 (2%)]	Loss: 2.293907
Train Epoch: 1 [1920/60000 (3%)]	Loss: 2.249022
Train Epoch: 1 [2560/60000 (4%)]	Loss: 2.242118
Train Epoch: 1 [3200/60000 (5%)]	Loss: 2.184256
Train Epoch: 1 [3840/60000 (6%)]	Loss: 2.234403
Train Epoch: 1 [4480/60000 (7%)]	Loss: 2.040562
Train Epoch: 1 [5120/60000 (9%)]	Loss: 1.957410
Train Epoch: 1 [5760/60000 (10%)]	Loss: 1.693862
Train Epoch: 1 [6400/60000 (11%)]	Loss: 1.873148
Train Epoch: 1 [7040/60000 (12%)]	Loss: 1.721508
Train Epoch: 1 [7680/60000 (13%)]	Loss: 1.881822
Train Epoch: 1 [8320/60000 (14%)]	Loss: 1.754513
Train Epoch: 1 [8960/60000 (15%)]	Loss: 1.787347
Train Epoch: 1 [9600/60000 (16%)]	Loss: 1.746670
Train Epoch: 1 [10240/60000 (17%)]	Loss: 1.524373
Train Epoch: 1 [10880/60000 (18%)]	Loss: 1.684197
Train Epoch: 1 [11520/60000 (19%)]	Loss: 1.688373
Train Epoch: 1 [12